# Transformer Fine-Tuning — distilbert-base-uncased

Replaces TF-IDF + classical ML with fine-tuned transformer for all three classifiers.

| Improvement | Detail |
|-------------|--------|
| **Backbone** | `distilbert-base-uncased` (66M params, ~5x faster than roberta) |
| **Urgency** | 4-class — **synthetic content-based labels** (not heuristic) |
| **Binary** | 2-class classification head |
| **Essential** | Multi-label classification (**10 labels**, 3 rare removed) |
| **Rare removed** | `missing_people` (86.9:1), `search_and_rescue` (35.2:1), `transport` (20.8:1) |
| **Urgency labels** | Synthetic: text content + category signals, replaces heuristic score formula |

---

## 0 · Setup

In [ ]:
# Colab: install if needed
# !pip install transformers datasets torch scikit-learn pandas matplotlib seaborn

import re, pickle, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    hamming_loss, jaccard_score,
    classification_report, confusion_matrix,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

# Paths
ROOT      = Path("..").resolve()           # Colab: adjust to your path
DATA_DIR  = ROOT / "data"
MODEL_DIR = ROOT / "models"
OUT_DIR   = ROOT / "outputs"
MODEL_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    print("Apple Silicon GPU (MPS) detected")

In [ ]:
ALL_ESSENTIAL = [
    "medical_help", "medical_products", "death",
    "floods", "storm", "earthquake",
    "water", "food", "shelter",
    "aid_related",
    "search_and_rescue", "transport", "missing_people",
]

ESSENTIAL_CATEGORIES = [
    "medical_help", "medical_products", "death",
    "floods", "storm", "earthquake",
    "water", "food", "shelter",
    "aid_related",
]
REMOVED_RARE = [c for c in ALL_ESSENTIAL if c not in ESSENTIAL_CATEGORIES]
print(f"Essential labels: {len(ESSENTIAL_CATEGORIES)} (removed rare: {REMOVED_RARE})")

URGENCY_WEIGHTS = {
    "death": 4, "medical_help": 3, "medical_products": 3,
    "search_and_rescue": 3, "missing_people": 3,
    "water": 2, "food": 2, "shelter": 2,
    "floods": 2, "earthquake": 2, "storm": 2, "fire": 2,
    "infrastructure_related": 1, "transport": 1, "buildings": 1,
    "electricity": 1, "other_aid": 1, "refugees": 1, "direct_report": 1,
}
URGENCY_THRESHOLDS = {0: 0, 1: 1, 4: 2, 7: 3}
URGENCY_LABEL_NAMES = {0: "low", 1: "medium", 2: "high", 3: "critical"}
UL = list(URGENCY_LABEL_NAMES.values())

BASELINE = {
    "urg_f1mac": 0.5218, "urg_acc": 0.5976, "urg_f1wt": 0.6059,
    "urg_per": [0.7760, 0.5121, 0.3658, 0.4334],
    "bin_f1": 0.8278, "bin_auc": 0.8717, "bin_acc": 0.7913,
    "bin_rec": 0.9028, "bin_prec": 0.7644,
    "ess_f1mic": 0.6991, "ess_f1mac": 0.5978,
    "ess_hl": 0.0583, "ess_jac": 0.3662,
}
print("Constants loaded.")

---
## 1 · Synthetic Urgency Labeling

Replaces the heuristic score formula with content-aware urgency scoring.
Uses **text signals** (keywords, distress patterns) + **category activations** + **genre**.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def parse_categories(cat_series):
    split = cat_series.str.split(";", expand=True)
    col_names = split.iloc[0].apply(lambda x: x.rsplit("-", 1)[0])
    split.columns = col_names
    for col in split.columns:
        split[col] = split[col].apply(lambda x: int(str(x).rsplit("-", 1)[-1])).clip(0, 1)
    return split.astype(np.int8)


def text_urgency_score(text, row):
    """Content-aware urgency: text signals + category signals + genre."""
    t = text.lower()
    score = 0.0

    # ── CRITICAL signals (life-threatening) ──
    if any(w in t for w in ["dying","dead","death","kill","killed","trapped","buried"]):
        score += 4.0
    if any(p in t for p in ["nothing to eat","nothing to drink","starving","we are dying",
                             "no food no water","going to die","children dying"]):
        score += 4.0
    if row.get("death", 0) == 1:
        score += 3.0

    # ── HIGH signals (urgent distress) ──
    high_w = ["emergency","urgent","desperately","destroyed","collapsed","injured",
              "wounded","critical","flooded","rubble","medical","hospital"]
    score += min(sum(1 for w in high_w if w in t) * 1.2, 3.0)
    high_p = ["need help","please help","need rescue","need medical","people dead",
              "need water","need food","need shelter","need tent","no water","no food",
              "no shelter","dire need","immediate need","500 people","hundreds","many people"]
    score += min(sum(1 for p in high_p if p in t) * 1.2, 3.0)
    if row.get("medical_help", 0) == 1:        score += 1.5
    if row.get("search_and_rescue", 0) == 1:  score += 1.5
    if row.get("missing_people", 0) == 1:     score += 1.5

    # ── MEDIUM signals (requests/needs) ──
    med_w = ["need","want","require","looking for","searching","missing","lost",
             "hungry","thirsty","homeless","food","water","tent","supply","help",
             "assist","rescue","shelter","medicine"]
    score += min(sum(1 for w in med_w if w in t) * 0.6, 2.5)
    if row.get("infrastructure_related", 0) == 1:
        score += 0.5

    # ── Genre boost (direct reports from victims are more urgent) ──
    if row.get("genre", "") == "direct_report":
        score += 1.5

    # ── Negative signals (lower urgency) ──
    low_p = ["thank you","thank god","we are fine","we are safe","we are okay",
             "just update","weather","announcement"]
    score -= sum(1 for p in low_p if p in t) * 1.0

    return max(0.0, min(12.0, score))


def score_to_level(score, thresholds):
    """Map continuous score → 0/1/2/3 using percentile thresholds."""
    if score <= thresholds["q33"]:  return 0
    elif score <= thresholds["q66"]: return 1
    elif score <= thresholds["q90"]: return 2
    else:                            return 3


def compute_heuristic_level(row):
    """Original heuristic: score = sum(category × weight), then thresholds."""
    score = sum(row.get(c, 0) * w for c, w in URGENCY_WEIGHTS.items())
    if score == 0:   return 0
    elif score <= 3: return 1
    elif score <= 6: return 2
    else:            return 3

In [ ]:
messages   = pd.read_csv(DATA_DIR / "disaster_messages.csv")
categories = pd.read_csv(DATA_DIR / "disaster_categories.csv")

df = messages.merge(categories, on="id")
df = df.drop_duplicates(subset="id").reset_index(drop=True)
cat_df = parse_categories(df["categories"])
df = pd.concat([df.drop(columns=["categories"]), cat_df], axis=1)
df["message_clean"] = df["message"].apply(clean_text)

# Aid-related flag
df["aid_related"] = (
    (df.get("aid_related", pd.Series(0, index=df.index)) == 1) |
    (df.get("request",     pd.Series(0, index=df.index)) == 1) |
    (df.get("other_aid",   pd.Series(0, index=df.index)) == 1)
).astype(np.int8)

# ── Heuristic labels (old) ──
df["urgency_heuristic"] = df.apply(compute_heuristic_level, axis=1)

# ── Synthetic labels (new) ──
df["input_text"] = df["genre"].fillna("unknown") + " " + df["message_clean"]
df["synth_score"] = df.apply(lambda r: text_urgency_score(r["message_clean"], r), axis=1)

# Compute percentile thresholds for level mapping
synth_thresh = {
    "q33": df["synth_score"].quantile(0.35),
    "q66": df["synth_score"].quantile(0.65),
    "q90": df["synth_score"].quantile(0.88),
}
df["urgency_level"] = df["synth_score"].apply(lambda s: score_to_level(s, synth_thresh))

noise_cols = {"related","request","offer","direct_report","child_alone","tools","shops"}
meaningful = [c for c in cat_df.columns if c not in noise_cols]
df["is_disaster"] = (
    (df["related"] == 1) & (df["meaningful" if "meaningful" in df.columns else meaningful].sum(axis=1) >= 1)
).astype(np.int8)

print(f"Dataset: {df.shape}")
print(f"Synthetic thresholds: low<={synth_thresh['q33']:.1f}  med<={synth_thresh['q66']:.1f}  high<={synth_thresh['q90']:.1f}  critical>{synth_thresh['q90']:.1f}")
print(f"Urgency (synthetic): {df['urgency_level'].value_counts().sort_index().to_dict()}")
print(f"Disaster: {df['is_disaster'].value_counts().to_dict()}")

In [ ]:
# Compare heuristic vs synthetic distributions
print("=== Heuristic vs Synthetic urgency distributions ===")
print(f"{'Level':<10} {'Heuristic':>12} {'Synthetic':>12} {'Δ%':>8}")
print("-" * 44)
for lev in [0, 1, 2, 3]:
    hc = (df["urgency_heuristic"] == lev).sum()
    sc = (df["urgency_level"] == lev).sum()
    hp = 100 * hc / len(df)
    sp = 100 * sc / len(df)
    print(f"Level {lev:<4} {hc:>12} ({hp:4.1f}%) {sc:>12} ({sp:4.1f}%) {sp-hp:>+6.1f}%")

agree = (df["urgency_heuristic"] == df["urgency_level"]).sum()
print(f"\nAgreement: {agree}/{len(df)} ({100*agree/len(df):.1f}%)")

---
## 2 · Dataset & Tokenizer

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3          # Colab GPU: keep 3. CPU only: reduce to 1.
LR = 2e-5

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class MultiLabelDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

def train_epoch(model, dl, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total_loss += out.loss.item()
    return total_loss / len(dl)

def eval_clf(model, dl, device):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in dl:
            labs = batch.pop("labels").to(device)
            out = model(**{k: v.to(device) for k, v in batch.items()})
            preds.extend(torch.argmax(out.logits, 1).cpu().numpy())
            labels.extend(labs.cpu().numpy())
    return np.array(preds), np.array(labels)

def eval_ml(model, dl, device, thr=0.5):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in dl:
            labs = batch.pop("labels").to(device)
            out = model(**{k: v.to(device) for k, v in batch.items()})
            preds.extend((torch.sigmoid(out.logits) >= thr).int().cpu().numpy())
            labels.extend(labs.cpu().numpy())
    return np.array(preds), np.array(labels)

print(f"Model: {MODEL_NAME}  MaxLen: {MAX_LEN}  Batch: {BATCH_SIZE}  Epochs: {EPOCHS}")

---
## 3 · MODEL 1 — Urgency Level (distilbert, synthetic labels)

In [ ]:
X_u = df["input_text"].tolist()
y_u = df["urgency_level"].values
Xtr_u, Xte_u, ytr_u, yte_u = train_test_split(X_u, y_u, test_size=0.2, random_state=42, stratify=y_u)

train_dl_u = DataLoader(TextDataset(Xtr_u, ytr_u, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
test_dl_u  = DataLoader(TextDataset(Xte_u, yte_u, tokenizer, MAX_LEN), batch_size=BATCH_SIZE)

model_u = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=4, problem_type="single_label_classification").to(DEVICE)
opt_u = AdamW(model_u.parameters(), lr=LR, weight_decay=0.01)
sch_u = get_linear_schedule_with_warmup(opt_u, num_warmup_steps=len(train_dl_u)*EPOCHS//10,
                                         num_training_steps=len(train_dl_u)*EPOCHS)

print(f"Training urgency ({EPOCHS} epochs, synthetic labels) …")
for ep in range(EPOCHS):
    t0 = time.time()
    loss = train_epoch(model_u, train_dl_u, opt_u, sch_u, DEVICE)
    preds, labels = eval_clf(model_u, test_dl_u, DEVICE)
    f1 = f1_score(labels, preds, average="macro")
    print(f"  Ep {ep+1}/{EPOCHS}  loss={loss:.4f}  F1 macro={f1:.4f}  ({time.time()-t0:.0f}s)")

yp_u = preds
urg_met = {
    "acc": round(accuracy_score(yte_u, yp_u), 4),
    "f1mac": round(f1_score(yte_u, yp_u, average="macro"), 4),
    "f1wt": round(f1_score(yte_u, yp_u, average="weighted"), 4),
    "per": [round(f1_score(yte_u==i, yp_u==i, zero_division=0), 4) for i in range(4)],
}
print(f"\n✓ Urgency F1 macro={urg_met['f1mac']:.4f}  acc={urg_met['acc']:.4f}")

In [ ]:
print(classification_report(yte_u, yp_u, target_names=UL))

cm = confusion_matrix(yte_u, yp_u)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=UL, yticklabels=UL, ax=ax)
ax.set_title("Urgency — Confusion Matrix (distilbert, synthetic labels)", fontsize=13, pad=12)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(OUT_DIR / "transformer_confusion_urgency.png", dpi=150)
plt.show()

---
## 4 · MODEL 2 — Disaster Binary (distilbert)

In [ ]:
X_b = df["input_text"].tolist()
y_b = df["is_disaster"].values
Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(X_b, y_b, test_size=0.2, random_state=42, stratify=y_b)

train_dl_b = DataLoader(TextDataset(Xtr_b, ytr_b, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
test_dl_b  = DataLoader(TextDataset(Xte_b, yte_b, tokenizer, MAX_LEN), batch_size=BATCH_SIZE)

model_b = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, problem_type="single_label_classification").to(DEVICE)
opt_b = AdamW(model_b.parameters(), lr=LR, weight_decay=0.01)
sch_b = get_linear_schedule_with_warmup(opt_b, num_warmup_steps=len(train_dl_b)*EPOCHS//10,
                                         num_training_steps=len(train_dl_b)*EPOCHS)

print(f"Training binary ({EPOCHS} epochs) …")
for ep in range(EPOCHS):
    t0 = time.time()
    loss = train_epoch(model_b, train_dl_b, opt_b, sch_b, DEVICE)
    preds, labels = eval_clf(model_b, test_dl_b, DEVICE)
    f1 = f1_score(labels, preds, average="binary")
    print(f"  Ep {ep+1}/{EPOCHS}  loss={loss:.4f}  F1={f1:.4f}  ({time.time()-t0:.0f}s)")

yp_b = preds
bin_met = {
    "acc":  round(accuracy_score(yte_b, yp_b), 4),
    "f1":   round(f1_score(yte_b, yp_b, average="binary"), 4),
    "prec": round(precision_score(yte_b, yp_b, zero_division=0), 4),
    "rec":  round(recall_score(yte_b, yp_b, zero_division=0), 4),
}
print(f"\n✓ Binary F1={bin_met['f1']:.4f}  acc={bin_met['acc']:.4f}  prec={bin_met['prec']:.4f}  rec={bin_met['rec']:.4f}")

In [ ]:
print(classification_report(yte_b, yp_b, target_names=["not_disaster", "disaster"]))

---
## 5 · MODEL 3 — Essential Categories (distilbert, 10 labels)

Rare categories removed: `missing_people`, `search_and_rescue`, `transport`

In [ ]:
X_e = df["input_text"].tolist()
y_e = df[ESSENTIAL_CATEGORIES].values.astype("float32")
Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(X_e, y_e, test_size=0.2, random_state=42)

train_dl_e = DataLoader(MultiLabelDataset(Xtr_e, ytr_e, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
test_dl_e  = DataLoader(MultiLabelDataset(Xte_e, yte_e, tokenizer, MAX_LEN), batch_size=BATCH_SIZE)

model_e = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(ESSENTIAL_CATEGORIES), problem_type="multi_label_classification").to(DEVICE)
opt_e = AdamW(model_e.parameters(), lr=LR, weight_decay=0.01)
sch_e = get_linear_schedule_with_warmup(opt_e, num_warmup_steps=len(train_dl_e)*EPOCHS//10,
                                         num_training_steps=len(train_dl_e)*EPOCHS)
criterion_e = torch.nn.BCEWithLogitsLoss()

print(f"Training essential ({EPOCHS} epochs, {len(ESSENTIAL_CATEGORIES)} labels) …")
for ep in range(EPOCHS):
    t0 = time.time()
    model_e.train()
    total_loss = 0
    for batch in train_dl_e:
        labs = batch.pop("labels").to(DEVICE)
        out = model_e(**{k: v.to(DEVICE) for k, v in batch.items()})
        loss = criterion_e(out.logits, labs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_e.parameters(), 1.0)
        opt_e.step(); sch_e.step(); opt_e.zero_grad()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_dl_e)
    preds, labels = eval_ml(model_e, test_dl_e, DEVICE)
    f1m = f1_score(labels, preds, average="micro", zero_division=0)
    f1mac = f1_score(labels, preds, average="macro", zero_division=0)
    print(f"  Ep {ep+1}/{EPOCHS}  loss={avg_loss:.4f}  F1 micro={f1m:.4f}  F1 macro={f1mac:.4f}  ({time.time()-t0:.0f}s)")

yp_e, yt_e = preds, labels
ess_met = {
    "f1mic": round(f1_score(yt_e, yp_e, average="micro",    zero_division=0), 4),
    "f1mac": round(f1_score(yt_e, yp_e, average="macro",    zero_division=0), 4),
    "f1wt":  round(f1_score(yt_e, yp_e, average="weighted", zero_division=0), 4),
    "hl":    round(hamming_loss(yt_e, yp_e), 4),
    "jac":   round(jaccard_score(yt_e, yp_e, average="samples", zero_division=0), 4),
}
print(f"\n✓ Essential F1 micro={ess_met['f1mic']:.4f}  F1 macro={ess_met['f1mac']:.4f}  HL={ess_met['hl']:.4f}")

In [ ]:
per_f1 = [round(f1_score(yt_e[:, i], yp_e[:, i], zero_division=0), 4) for i in range(len(ESSENTIAL_CATEGORIES))]
for cat, f1 in sorted(zip(ESSENTIAL_CATEGORIES, per_f1), key=lambda x: -x[1]):
    bar = "█" * int(f1 * 20) + "░" * (20 - int(f1 * 20))
    print(f"  {cat:<25} {bar}  {f1:.4f}")

order_idx = np.argsort(per_f1)
cats_ord = [ESSENTIAL_CATEGORIES[i] for i in order_idx]
f1_ord = [per_f1[i] for i in order_idx]
colors = ["#2196F3" if s >= 0.7 else "#FF9800" if s >= 0.5 else "#F44336" for s in f1_ord]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cats_ord, f1_ord, color=colors, edgecolor="white")
ax.axvline(0.7, color="green",  linestyle="--", lw=1.2, label="0.70")
ax.axvline(0.5, color="orange", linestyle="--", lw=1.2, label="0.50")
for bar, score in zip(bars, f1_ord):
    ax.text(score + 0.01, bar.get_y() + bar.get_height()/2, f"{score:.3f}", va="center", fontsize=10)
ax.set_xlabel("F1 Score"); ax.set_xlim(0, 1.05)
ax.set_title("Essential Categories — Per-Label F1 (distilbert, 10 labels)", fontsize=13, pad=12)
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "transformer_label_f1_essential.png", dpi=150)
plt.show()

---
## 6 · Save Models

In [ ]:
model_u.save_pretrained(MODEL_DIR / "transformer_urgency")
tokenizer.save_pretrained(MODEL_DIR / "transformer_urgency")
model_b.save_pretrained(MODEL_DIR / "transformer_binary")
tokenizer.save_pretrained(MODEL_DIR / "transformer_binary")
model_e.save_pretrained(MODEL_DIR / "transformer_essential")
tokenizer.save_pretrained(MODEL_DIR / "transformer_essential")
with open(MODEL_DIR / "transformer_essential_meta.pkl", "wb") as f:
    pickle.dump({"categories": ESSENTIAL_CATEGORIES, "threshold": 0.5}, f)
print("✓ All transformer models saved.")

---
## 7 · Performance Review — TF-IDF vs distilbert

In [ ]:
def row(label, bv, av, hb=True):
    imp  = (av > bv) if hb else (av < bv)
    same = abs(av - bv) < 0.0001
    icon = "➖" if same else ("✅" if imp else "🔴")
    d    = av - bv
    ds   = (f"+{d:.4f}" if d >= 0 else f"{d:.4f}")
    return f"  {label:<32} {bv:.4f}  →  {av:.4f}  {ds:>9}  {icon}"

print("═" * 68)
print("  TF-IDF (before)  vs  distilbert-base-uncased (after)")
print("═" * 68)

In [ ]:
# MODEL 1
print(f"\n  ┌── MODEL 1: Urgency Level (synthetic labels) ─────────────────┐")
print(f"  │  distilbert-base-uncased, {EPOCHS} epoch(s), lr={LR}                │")
print(f"  └──────────────────────────────────────────────────────────────┘")
print(f"  {'Metric':<32} {'TF-IDF':>7}    {'distilbert':>11}   {'Δ':>9}")
print(f"  {'-'*62}")
print(row("Accuracy",   BASELINE["urg_acc"],  urg_met["acc"]))
print(row("F1 Macro ★", BASELINE["urg_f1mac"], urg_met["f1mac"]))
for i, n in enumerate(UL):
    print(row(f"  [{n}]", BASELINE["urg_per"][i], urg_met["per"][i]))

# MODEL 2
print(f"\n  ┌── MODEL 2: Disaster Binary ───────────────────────────────────┐")
print(f"  │  distilbert-base-uncased, {EPOCHS} epoch(s), lr={LR}                │")
print(f"  └──────────────────────────────────────────────────────────────┘")
print(f"  {'Metric':<32} {'TF-IDF':>7}    {'distilbert':>11}   {'Δ':>9}")
print(f"  {'-'*62}")
print(row("Accuracy",    BASELINE["bin_acc"], bin_met["acc"]))
print(row("F1 ★",        BASELINE["bin_f1"],  bin_met["f1"]))
print(row("Precision",   BASELINE["bin_prec"], bin_met["prec"]))
print(row("Recall",      BASELINE["bin_rec"], bin_met["rec"]))

# MODEL 3
print(f"\n  ┌── MODEL 3: Essential Categories ({len(ESSENTIAL_CATEGORIES)} labels) ──────────────┐")
print(f"  │  distilbert-base-uncased, {EPOCHS} epoch(s), lr={LR}                │")
print(f"  │  Rare removed: {', '.join(REMOVED_RARE):<44}│")
print(f"  └──────────────────────────────────────────────────────────────┘")
print(f"  {'Metric':<32} {'TF-IDF':>7}    {'distilbert':>11}   {'Δ':>9}")
print(f"  {'-'*62}")
print(row("F1 Micro ★",  BASELINE["ess_f1mic"], ess_met["f1mic"]))
print(row("F1 Macro",    BASELINE["ess_f1mac"], ess_met["f1mac"]))
print(row("Hamming Loss", BASELINE["ess_hl"],  ess_met["hl"], hb=False))
print(row("Jaccard",     BASELINE["ess_jac"],  ess_met["jac"]))

In [ ]:
urg_ok = urg_met["f1mac"] > BASELINE["urg_f1mac"]
bin_ok = bin_met["f1"]    > BASELINE["bin_f1"]
ess_ok = ess_met["f1mic"] > BASELINE["ess_f1mic"]
total  = sum([urg_ok, bin_ok, ess_ok])

print(f"\n  {'═'*62}")
print(f"  VERDICT: {total}/3 models improved with distilbert + synthetic labels")
print(f"  {'─'*62}")
def vline(name, ok, bv, av):
    s = f"{bv:.4f} → {av:.4f} ({'+' if av>=bv else ''}{av-bv:.4f})"
    return f"  {'✅' if ok else '🔴'} {name:<30} {s}"
print(vline("Urgency  (F1 macro)",  urg_ok, BASELINE["urg_f1mac"], urg_met["f1mac"]))
print(vline("Binary   (F1)",        bin_ok, BASELINE["bin_f1"],    bin_met["f1"]))
print(vline("Ess.Cats (F1 micro)",  ess_ok, BASELINE["ess_f1mic"], ess_met["f1mic"]))
print(f"  {'═'*62}")

In [ ]:
# Dashboard
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("TF-IDF vs distilbert-base-uncased (synthetic urgency labels)", fontsize=14, fontweight="bold")

# Urgency
labs_u = ["Accuracy", "F1 Macro", "F1 Weighted"]
bv_u = [BASELINE["urg_acc"], BASELINE["urg_f1mac"], BASELINE["urg_f1wt"]]
av_u = [urg_met["acc"], urg_met["f1mac"], urg_met["f1wt"]]
x = np.arange(len(labs_u))
axes[0].bar(x - 0.15, bv_u, 0.3, label="TF-IDF", color="#90CAF9", edgecolor="white")
axes[0].bar(x + 0.15, av_u, 0.3, label="distilbert", color="#2196F3", edgecolor="white")
axes[0].set_xticks(x); axes[0].set_xticklabels(labs_u)
axes[0].set_title("Urgency"); axes[0].set_ylabel("Score"); axes[0].legend(fontsize=9)

# Binary
labs_b = ["Accuracy", "F1", "Precision", "Recall"]
bv_b = [BASELINE["bin_acc"], BASELINE["bin_f1"], BASELINE["bin_prec"], BASELINE["bin_rec"]]
av_b = [bin_met["acc"], bin_met["f1"], bin_met["prec"], bin_met["rec"]]
x2 = np.arange(len(labs_b))
axes[1].bar(x2 - 0.15, bv_b, 0.3, label="TF-IDF", color="#A5D6A7", edgecolor="white")
axes[1].bar(x2 + 0.15, av_b, 0.3, label="distilbert", color="#4CAF50", edgecolor="white")
axes[1].set_xticks(x2); axes[1].set_xticklabels(labs_b)
axes[1].set_title("Binary"); axes[1].set_ylabel("Score"); axes[1].legend(fontsize=9)

# Essential
labs_e = ["F1 Micro", "F1 Macro", "Hamming Loss\n(inv)", "Jaccard"]
bv_e = [BASELINE["ess_f1mic"], BASELINE["ess_f1mac"], 1 - BASELINE["ess_hl"], BASELINE["ess_jac"]]
av_e = [ess_met["f1mic"], ess_met["f1mac"], 1 - ess_met["hl"], ess_met["jac"]]
x3 = np.arange(len(labs_e))
axes[2].bar(x3 - 0.15, bv_e, 0.3, label="TF-IDF", color="#FFCC80", edgecolor="white")
axes[2].bar(x3 + 0.15, av_e, 0.3, label="distilbert", color="#FF9800", edgecolor="white")
axes[2].set_xticks(x3); axes[2].set_xticklabels(labs_e)
axes[2].set_title("Essential (10 labels)"); axes[2].set_ylabel("Score"); axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "transformer_comparison_dashboard.png", dpi=150)
plt.show()

In [ ]:
rows = []
for metric, bv, av in [
    ("Urgency Accuracy",   BASELINE["urg_acc"],  urg_met["acc"]),
    ("Urgency F1 Macro",   BASELINE["urg_f1mac"], urg_met["f1mac"]),
    ("Binary F1",          BASELINE["bin_f1"],   bin_met["f1"]),
    ("Binary Recall",      BASELINE["bin_rec"],  bin_met["rec"]),
    ("Essential F1 Micro", BASELINE["ess_f1mic"], ess_met["f1mic"]),
    ("Essential Jaccard",  BASELINE["ess_jac"],  ess_met["jac"]),
]:
    rows.append({"metric":metric, "tfidf":bv, "distilbert":av,
                 "delta":round(av-bv,4), "improved":av>bv})
pd.DataFrame(rows).to_csv(OUT_DIR / "transformer_comparison.csv", index=False)
print("CSV → outputs/transformer_comparison.csv")

---
## 8 · Live Inference Demo

In [ ]:
def predict_clf(texts, model, tok):
    model.eval()
    enc = tok(texts, max_length=MAX_LEN, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = model(**{k: v.to(DEVICE) for k, v in enc.items()})
    return torch.argmax(out.logits, 1).cpu().numpy()

def predict_ml(texts, model, tok, thr=0.5):
    model.eval()
    enc = tok(texts, max_length=MAX_LEN, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = model(**{k: v.to(DEVICE) for k, v in enc.items()})
    return (torch.sigmoid(out.logits) >= thr).int().cpu().numpy()

demo = [
    ("People are trapped under collapsed buildings, we need rescue teams now.",    "direct"),
    ("Weather update — a cold front from Cuba could pass over Haiti tomorrow.",     "news"),
    ("We have no food or water in our shelter. Children are sick.",                 "direct"),
    ("Government announces new relief fund for flood victims.",                     "news"),
    ("Hospital completely destroyed by earthquake. Patients on the street.",        "direct"),
    ("Storm approaching the northern coast. Residents advised to evacuate.",        "news"),
    ("We are dying of hunger — 500 people in Delmas 19 need immediate help.",      "direct"),
    ("Missing: my sister Maryani, last seen near Petionville on Tuesday.",          "direct"),
]
X_demo = [f"{g} {clean_text(t)}" for t, g in demo]

urg_p = [URGENCY_LABEL_NAMES[p] for p in predict_clf(X_demo, model_u, tokenizer)]
bin_p = ["disaster" if p == 1 else "not_disaster" for p in predict_clf(X_demo, model_b, tokenizer)]
ess_p = predict_ml(X_demo, model_e, tokenizer)

print(f"{'#':>3}  {'DISASTER':^12}  {'URGENCY':^10}  ESSENTIAL CATEGORIES")
print("─" * 90)
for i, ((text, genre), dis, urg, cats) in enumerate(zip(demo, bin_p, urg_p, ess_p), 1):
    active = [ESSENTIAL_CATEGORIES[j] for j, v in enumerate(cats) if v == 1]
    d = "✓ YES" if dis == "disaster" else "✗ NO "
    s = text[:48] + "…" if len(text) > 48 else text
    print(f"{i:>3}. {d:^12}  {urg:^10}  {', '.join(active) or '—'}")
    print(f"     [{genre}] \"{s}\"")
    print()

---
## Notes

- **distilbert-base-uncased** is English-only. For Haitian Creole, use `xlm-roberta-base`.
- **Rare categories removed** (`missing_people`, `search_and_rescue`, `transport`) to maximize F1.
- **Synthetic urgency labels** use text content + category signals, not just heuristic category weights.
- **Colab tip:** Enable GPU for ~10x faster training (~3 min for 3 epochs).